In [3]:
# STEP 1: Install Libraries
!pip install -U langchain langchain-community langchain-text-splitters langgraph chromadb pypdf sentence-transformers -q


# STEP 2: Upload PDF
from google.colab import files

print("Please upload your customer support PDF file")
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]
print("Uploaded file:", pdf_path)


# STEP 3: Load PDF
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print("Total pages loaded:", len(documents))
print("Sample text:")
print(documents[0].page_content[:500])


# STEP 4: Split PDF into Chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Total chunks created:", len(chunks))
print("First chunk:")
print(chunks[0].page_content)


# STEP 5: Create Embeddings and Store in ChromaDB
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

retriever = vector_db.as_retriever(
    search_kwargs={"k": 3}
)

print("ChromaDB vector store created successfully")


# STEP 6: Answer Generation Function
def generate_answer(query, docs):
    if not docs:
        return "I could not find relevant information."

    context = docs[0].page_content

    # Clean formatting
    context = context.replace("  ", " ").strip()

    answer = f"""
Answer:

{context}

Summary:
- Refund allowed within 7 days
- Process takes 5–7 business days
- Cancellation only before shipping
"""

    return answer

# STEP 7: HITL Escalation Logic
def check_escalation(query, docs):
    escalation_keywords = [
        "angry",
        "complaint",
        "legal",
        "fraud",
        "urgent",
        "human",
        "manager",
        "not satisfied",
        "refund not received",
        "payment failed",
        "money deducted"
    ]

    query_lower = query.lower()

    if not docs:
        return True

    if any(keyword in query_lower for keyword in escalation_keywords):
        return True

    if len(docs[0].page_content.strip()) < 50:
        return True

    return False


# STEP 8: Create LangGraph State
from typing import TypedDict
from langgraph.graph import StateGraph, END

class SupportState(TypedDict):
    query: str
    docs: list
    answer: str
    escalation_required: bool
    final_response: str


# STEP 9: Processing Node
def processing_node(state: SupportState):
    query = state["query"]

    docs = retriever.invoke(query)
    answer = generate_answer(query, docs)
    escalation = check_escalation(query, docs)

    return {
        "query": query,
        "docs": docs,
        "answer": answer,
        "escalation_required": escalation,
        "final_response": ""
    }


# STEP 10: Output Node
def output_node(state: SupportState):
    if state["escalation_required"]:
        final_response = f"""
==============================
BOT RESPONSE
==============================

{state['answer']}

==============================
STATUS
==============================

Escalated to Human Support Agent.

Reason:
The query may be complex, urgent, emotional, or not fully answered using the knowledge base.

Human-in-the-Loop Message:
A human support agent should review this customer query and provide the final response.
"""
    else:
        final_response = f"""
==============================
BOT RESPONSE
==============================

{state['answer']}

==============================
STATUS
==============================

Answered by RAG Customer Support Assistant.
"""

    return {
        "query": state["query"],
        "docs": state["docs"],
        "answer": state["answer"],
        "escalation_required": state["escalation_required"],
        "final_response": final_response
    }


# STEP 11: Build LangGraph Workflow
workflow = StateGraph(SupportState)

workflow.add_node("process", processing_node)
workflow.add_node("output", output_node)

workflow.set_entry_point("process")
workflow.add_edge("process", "output")
workflow.add_edge("output", END)

app = workflow.compile()

print("LangGraph workflow created successfully")


# STEP 12: Run Assistant
while True:
    user_query = input("\nEnter your customer support query or type 'exit': ")

    if user_query.lower() == "exit":
        print("Thank you for using the RAG Customer Support Assistant.")
        break

    result = app.invoke({
        "query": user_query,
        "docs": [],
        "answer": "",
        "escalation_required": False,
        "final_response": ""
    })

    print(result["final_response"])

Please upload your customer support PDF file


Saving customer_support_faq.pdf to customer_support_faq.pdf
Uploaded file: customer_support_faq.pdf
Total pages loaded: 1
Sample text:
CUSTOMER  SUPPORT  FAQ   1.  Refund  Policy  Customers  can  request  a  refund  within  7  days  of  purchase.  Refunds  are  processed  within  5–7  business  days  after  approval.   2.  Order  Cancellation  Orders  can  be  cancelled  before  they  are  shipped.  Once  shipped,  cancellation  is  not  possible.   3.  Delivery  Information  Orders  are  delivered  within  3–5  business  days.  Delivery  time  may  vary  depending  on  location.   4.  Payment  Issues  If  payment  fails,  plea
Total chunks created: 3
First chunk:
CUSTOMER  SUPPORT  FAQ   1.  Refund  Policy  Customers  can  request  a  refund  within  7  days  of  purchase.  Refunds  are  processed  within  5–7  business  days  after  approval.   2.  Order  Cancellation  Orders  can  be  cancelled  before  they  are  shipped.  Once  shipped,  cancellation  is  not  possible.   3.  Deli

/tmp/ipykernel_19596/2301822153.py:45: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ChromaDB vector store created successfully
LangGraph workflow created successfully

Enter your customer support query or type 'exit': What is the refund policy?

BOT RESPONSE


Answer:

CUSTOMER SUPPORT FAQ  1. Refund Policy Customers can request a refund within 7 days of purchase. Refunds are processed within 5–7 business days after approval.  2. Order Cancellation Orders can be cancelled before they are shipped. Once shipped, cancellation is not possible.  3. Delivery Information Orders are delivered within 3–5 business days. Delivery time may vary depending on location.  4. Payment Issues If payment fails,

Summary:
- Refund allowed within 7 days
- Process takes 5–7 business days
- Cancellation only before shipping


STATUS

Answered by RAG Customer Support Assistant.


Enter your customer support query or type 'exit': How can I cancel my order?

BOT RESPONSE


Answer:

CUSTOMER SUPPORT FAQ  1. Refund Policy Customers can request a refund within 7 days of purchase. Refunds are proce